# 01: Binary Classification, Decision Trees, Random Forests & XGBoost

**Track 05: Classification & Tabular Gradient Boosting** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Comprehensive classification metrics (ROC-AUC, Precision, Recall, F1, Log-Loss), threshold tuning, Decision Trees, Ensemble Bagging, and XGBoost on Titanic survival.


## 1. Classification Metrics & Decision Boundary Theory
Ingest Titanic dataset and preprocess categorical and missing features.

In [ ]:
import os
import sys
from pathlib import Path

for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (p / "utils").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

from utils.data_loader import load_dataset

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import xgboost as xgb

df = load_dataset("titanic")

# Basic feature engineering
df["Sex"] = df["Sex"].map({"male": 0, "female": 1}).fillna(0)
df["Embarked"] = df["Embarked"].map({"S": 0, "C": 1, "Q": 2}).fillna(0)
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = df[features]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

## 2. Comparing Random Forest vs XGBoost
Train and benchmark tree ensemble models.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]
rf_preds = (rf_probs >= 0.5).astype(int)

xgb_clf = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, eval_metric="logloss", random_state=42)
xgb_clf.fit(X_train, y_train)
xgb_probs = xgb_clf.predict_proba(X_test)[:, 1]
xgb_preds = (xgb_probs >= 0.5).astype(int)

print("=== Benchmark Comparison ===")
print(f"Random Forest -> ROC-AUC: {roc_auc_score(y_test, rf_probs):.4f} | F1: {f1_score(y_test, rf_preds):.4f} | Accuracy: {accuracy_score(y_test, rf_preds):.4f}")
print(f"XGBoost       -> ROC-AUC: {roc_auc_score(y_test, xgb_probs):.4f} | F1: {f1_score(y_test, xgb_preds):.4f} | Accuracy: {accuracy_score(y_test, xgb_preds):.4f}")
print("\nXGBoost Confusion Matrix:")
print(confusion_matrix(y_test, xgb_preds))